# Task #47 — Tính class_weight, lưu tập train/test, so sánh phân phối nhãn trước/sau xử lý (Story #8, giai đoạn 2)

Dựng lại pipeline Task #46 (tự chứa lại từ đầu theo quy ước các notebook trước), tính `class_weight` làm kỹ thuật xử lý mất cân bằng mặc định (quyết định đã chốt — không resample dữ liệu, SMOTE để dành so sánh thực nghiệm ở Story #9), lưu `orders_features_train.csv`/`orders_features_test.csv`.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

df = pd.read_csv("../data/processed/orders_features.csv", low_memory=False)
df["is_delayed"] = df["is_delayed"].astype("boolean")

bool_cols = ["payment_has_boleto", "payment_has_credit_card", "payment_has_debit_card",
             "payment_has_not_defined", "payment_has_voucher", "items_multi_seller"]
for col in bool_cols:
    df[col] = df[col].astype("boolean")

dfa = df[df["is_delayed"].notna()].copy()
feature_cols = [c for c in dfa.columns if c not in ("order_id", "is_delayed")]
dfa_clean = dfa[~dfa[feature_cols].isna().any(axis=1)].copy()

X = dfa_clean.drop(columns=["is_delayed"])
y = dfa_clean["is_delayed"].astype(bool)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42,
)
print("Train:", X_train.shape, " Test:", X_test.shape)

Train: (77156, 76)  Test: (19289, 76)


## 1. So sánh phân phối nhãn trước/sau xử lý

class_weight **không resample** — số dòng và tỉ lệ vật lý giữ nguyên ở cả train/test. "Xử lý" thể hiện qua trọng số áp dụng khi huấn luyện mô hình (Story #9): nhân trọng số vào từng lớp cho ra phân phối "hiệu lực" cân bằng 50/50 mà không cần thêm/bớt dòng dữ liệu thật.

In [2]:
print("Truoc xu ly (toan tap, Phuong an A):")
print(df["is_delayed"].dropna().astype(bool).value_counts(normalize=True))

print("\nTrain (raw, chua xu ly):")
print(y_train.value_counts(normalize=True))
print("\nTest (raw, chua xu ly):")
print(y_test.value_counts(normalize=True))

classes = np.array([False, True])
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight_dict = {bool(c): float(w) for c, w in zip(classes, weights)}
print("\nclass_weight (balanced, tinh tren train):", class_weight_dict)

scale_pos_weight = (~y_train).sum() / y_train.sum()
print("scale_pos_weight (cho XGBoost, ty le am/duong):", round(scale_pos_weight, 4))

weighted_counts = y_train.value_counts() * pd.Series(class_weight_dict)
weighted_pct = (weighted_counts / weighted_counts.sum()).round(4)
print("\nSau xu ly (phan phoi 'hieu luc' khi ap trong so, khong doi so dong that):")
print(weighted_pct)

Truoc xu ly (toan tap, Phuong an A):
is_delayed
False    0.918871
True     0.081129
Name: proportion, dtype: float64

Train (raw, chua xu ly):
is_delayed
False    0.918853
True     0.081147
Name: proportion, dtype: float64

Test (raw, chua xu ly):
is_delayed
False    0.918866
True     0.081134
Name: proportion, dtype: float64

class_weight (balanced, tinh tren train): {False: 0.5441568516820651, True: 6.161635521482191}
scale_pos_weight (cho XGBoost, ty le am/duong): 11.3233

Sau xu ly (phan phoi 'hieu luc' khi ap trong so, khong doi so dong that):
is_delayed
False    0.5
True     0.5
dtype: float64


## 2. Lưu tập train/test

`data/processed/orders_features_train.csv` / `orders_features_test.csv`. Cùng gotcha CSV không giữ dtype (Task #32) — đọc lại phải ép kiểu như Task #45/#46.

In [3]:
train_df = X_train.copy()
train_df["is_delayed"] = y_train
test_df = X_test.copy()
test_df["is_delayed"] = y_test

train_df.to_csv("../data/processed/orders_features_train.csv", index=False)
test_df.to_csv("../data/processed/orders_features_test.csv", index=False)

print(f"Da luu train: {len(train_df)} dong vao orders_features_train.csv")
print(f"Da luu test: {len(test_df)} dong vao orders_features_test.csv")

Da luu train: 77156 dong vao orders_features_train.csv
Da luu test: 19289 dong vao orders_features_test.csv
